# LabelInspect - GPU access check

This checks the runtime, not the paper model. No private data or tokens are required.

**Kaggle:** import this notebook, select an available GPU in Settings > Accelerator, then run the cells. Complete account verification directly with Kaggle if required. **Colab fallback:** upload this notebook and choose a GPU runtime. Availability is controlled by the provider.

Only keep the GPU enabled while doing GPU work. Save `gpu_environment.json` after running.

In [ ]:
import json, platform, sys
from pathlib import Path
import torch
report = {'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__,
          'cuda_runtime': torch.version.cuda, 'cuda_available': torch.cuda.is_available()}
if torch.cuda.is_available():
    report['gpu'] = torch.cuda.get_device_name(0)
    report['gpu_vram_gib'] = torch.cuda.get_device_properties(0).total_memory / 2**30
print(json.dumps(report, indent=2))
assert torch.cuda.is_available(), 'No GPU is active. Select a GPU accelerator or check account access/quota.'


In [ ]:
# Small hardware check only: this is not DTU-Net and does not train on images.
torch.manual_seed(230224)
device = torch.device('cuda:0')
model = torch.nn.Conv2d(3, 3, kernel_size=3, padding=1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randn(2, 3, 64, 64, device=device)
target = torch.randn_like(x)
losses = []
for _ in range(3):
    optimizer.zero_grad(set_to_none=True)
    loss = torch.nn.functional.mse_loss(model(x), target)
    assert torch.isfinite(loss)
    loss.backward()
    assert all(torch.isfinite(p.grad).all() for p in model.parameters())
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
torch.cuda.synchronize()
report['small_optimization_check'] = 'passed'
report['check_losses'] = losses
out = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
path = out / 'gpu_environment.json'
path.write_text(json.dumps(report, indent=2))
print('GPU check passed. Saved:', path)
print('GPU:', report['gpu'])


## Next step
Keep the printed GPU name and the JSON report. The paper-model notebook will need a separately tested dependency set, data, and checkpoint/resume support. Passing this check is not proof that the paper configuration fits in GPU memory. Do not install the original requirements list blindly.

# LabelInspect - author model integration check

This runs the actual author DTU-Net (UDHVT in the repository), the custom Tsimplex generator, the author L2 noise loss, and a short author reconstruction path. It performs only **two optimizer updates on synthetic normal images**. It does not train a useful anomaly detector or reproduce reported metrics.

**Requirements:** the verified T4 runtime, Internet access for five pinned GitHub source files, and the already-installed PyTorch/timm/einops/Numba stack. This notebook does not reinstall PyTorch. Run on one GPU. The first Numba compilation can take several minutes.

Source: Kumar et al., WACV 2025; https://github.com/MAXNORM8650/Annotsim, commit `dc4a9bd2a2a5b1c31223daab4bdfea3f6a5b2990`. Import fixes, the tuple-output adapter and selected configuration are recorded in the report.

In [ ]:
from pathlib import Path
import json, sys, subprocess, importlib.metadata
import torch
assert torch.cuda.is_available(), 'Select a GPU accelerator first.'
for name in ['timm','einops','numba','numpy','Pillow']:
    print(name, importlib.metadata.version(name))
print('GPU:', torch.cuda.get_device_name(0))
work = Path('/kaggle/working/labelinspect') if Path('/kaggle/working').exists() else Path.cwd() / 'labelinspect'
work.mkdir(parents=True, exist_ok=True)


In [ ]:
script = work / 'author_smoke.py'
script.write_text('"""Run a bounded integration check of the author DTU-Net and Tsimplex code.\n\nThis is not training for anomaly detection, not a reproduced result, and not a\nperformance comparison. Downloads only five source files from a pinned commit.\n"""\nfrom __future__ import annotations\nimport argparse\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nimport types\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\nCOMMIT = \'dc4a9bd2a2a5b1c31223daab4bdfea3f6a5b2990\'\nBASE_URL = f\'https://raw.githubusercontent.com/MAXNORM8650/Annotsim/{COMMIT}/\'\nFILES = [\'src/models/UModels/UDHVT.py\',\'GaussianDiffusion.py\',\n         \'utils/Simplex/constants.py\',\'utils/Simplex/internals.py\',\'utils/Simplex/noise.py\']\nEXPECTED = {\n \'utils/Simplex/constants.py\':\'52bf6ba3e2c0d386fa420382de380093a8dd61f488765cb812b13e25d0be7294\',\n \'utils/Simplex/internals.py\':\'ef67562885dcfe3356acd97784fe10660bf21238be7bcc608e86053c529fd61a\',\n \'src/models/UModels/UDHVT.py\':\'f7303c4dd228a3f5e1ab98d16fe1db7c7abfecfa97f683e89449128c5a03a4c2\',\n \'GaussianDiffusion.py\':\'cdf7a2143a441d20a3250458c0683c53ac1484ef8f0d0927831e66a34b52ec9a\',\n \'utils/Simplex/noise.py\':\'d114b6898369a0299e48f95fe4165fb3d587dcb8d7e257b58aef02077b6249d3\',\n}\n\n\ndef fetch_sources(root):\n    hashes={}\n    for name in FILES:\n        destination=root/name\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        if not destination.exists():\n            print(\'Downloading\',name,flush=True)\n            last_error=None\n            for attempt in range(1,4):\n                try:\n                    with urllib.request.urlopen(BASE_URL+name,timeout=45) as response:\n                        payload=response.read()\n                    break\n                except urllib.error.URLError as exc:\n                    last_error=exc\n                    print(f\'Network attempt {attempt}/3 failed: {exc}\',flush=True)\n                    if attempt < 3:time.sleep(2*attempt)\n            else:\n                raise RuntimeError(\n                    \'Could not download the pinned public author files. In Kaggle, \'\n                    \'open Settings, turn Internet on, then rerun this cell.\'\n                ) from last_error\n            if name in EXPECTED and hashlib.sha256(payload).hexdigest()!=EXPECTED[name]:\n                raise ValueError(f\'Inspected-source hash mismatch: {name}; stop and review this revision.\')\n            destination.write_bytes(payload)\n        digest=hashlib.sha256(destination.read_bytes()).hexdigest()\n        if name in EXPECTED and digest!=EXPECTED[name]:\n            raise ValueError(f\'Cached-source hash mismatch: {name}; use a fresh cache after review.\')\n        hashes[name]=digest\n    return hashes\n\n\ndef load_module(name,path):\n    spec=importlib.util.spec_from_file_location(name,path)\n    module=importlib.util.module_from_spec(spec)\n    sys.modules[name]=module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef load_author_components(source_root,output):\n    import numpy as np\n    import torch\n    import torch.nn as nn\n    # Namespace isolation avoids importing the repository\'s unrelated experiments.\n    package=types.ModuleType(\'labelinspect_author_simplex\')\n    package.__path__=[str(source_root/\'utils/Simplex\')]\n    sys.modules[package.__name__]=package\n    noise=load_module(package.__name__+\'.noise\',source_root/\'utils/Simplex/noise.py\')\n\n    original=(source_root/\'src/models/UModels/UDHVT.py\').read_text(encoding=\'utf8\')\n    replacements={\n      \'from torchvision import models\':\'# Removed unused torchvision.models import.\',\n      \'from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD, IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD\':\'# Removed unused timm image constants.\',\n      \'from timm.models.helpers import build_model_with_cfg, named_apply, adapt_input_conv\':\'from timm.models._manipulate import named_apply\',\n      \'from timm.models.layers import trunc_normal_, lecun_normal_, to_2tuple\':\'from timm.layers import trunc_normal_, lecun_normal_, to_2tuple\',\n      \'from timm.models.registry import register_model\':\'# Removed unused timm registry import.\',\n    }\n    for old,new in replacements.items():\n        if original.count(old)!=1:raise ValueError(\'Compatibility patch no longer matches the inspected source: \'+old)\n        original=original.replace(old,new)\n    patched=output/\'author_UDHVT_compat.py\'\n    patched.write_text(\'import numpy as np\\n\'+original,encoding=\'utf8\')\n    model_module=load_module(\'labelinspect_author_model\',patched)\n\n    # Keep the author definitions, including its variance convention. Avoid the\n    # top-level imports for unused image losses, plotting, datasets and backbones.\n    tree=ast.parse((source_root/\'GaussianDiffusion.py\').read_text(encoding=\'utf8\'))\n    wanted={\'get_beta_schedule\',\'extract\',\'mean_flat\',\'generate_simplex_4noise\',\'GaussianDiffusionModel\'}\n    selected=[node for node in tree.body if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in wanted]\n    if {node.name for node in selected}!=wanted:raise ValueError(\'Required author diffusion definitions are missing\')\n    reduced=ast.Module(body=selected,type_ignores=[])\n    namespace={\'np\':np,\'torch\':torch,\'nn\':nn,\'OpenSimplex\':noise.OpenSimplex}\n    exec(compile(reduced,\'author_diffusion_l2_subset.py\',\'exec\'),namespace)\n    (output/\'author_diffusion_l2_subset.py\').write_text(ast.unparse(reduced),encoding=\'utf8\')\n\n    class NoisePredictionAdapter(nn.Module):\n        def __init__(self,backbone):super().__init__();self.backbone=backbone\n        def forward(self,x,t,y=None):\n            if y is not None:raise ValueError(\'This initial adapter supports the normal-only, unconditioned path\')\n            result=self.backbone(x,t,y=None)\n            prediction=result[0] if isinstance(result,tuple) else result\n            if prediction.shape!=x.shape:raise ValueError(\'Noise prediction does not match input shape\')\n            return prediction\n\n    return model_module,namespace,NoisePredictionAdapter,{\n      \'imports\':replacements,\'extra_import\':\'numpy for the author PositionalEmbedding helper\',\n      \'adapter\':\'Select tuple element 0; preserve the backbone computation.\',\n      \'diffusion_loading\':\'AST-load only the author definitions required for Gaussian/Tsimplex L2 and sampling; other losses are not supported.\',\n      \'sampling\':\'Pass denoise_fn=noise_fn so reverse steps use configured O/mu/p instead of the author alternate branch defaults.\',\n      \'calling_convention\':\'Set author diffusion train=False to select model(x,t,y=lab) during sampling. This is a dispatch flag; the model is explicitly switched with model.train()/eval().\',\n    }\n\n\ndef synthetic_batch(size,batch,device):\n    import numpy as np\n    import torch\n    from PIL import Image,ImageDraw,ImageFont\n    im=Image.new(\'L\',(size,size),235);draw=ImageDraw.Draw(im)\n    try:font=ImageFont.truetype(\'DejaVuSans.ttf\',20)\n    except OSError:font=ImageFont.load_default(size=20)\n    draw.rectangle((12,12,size-12,size-12),outline=20,width=2)\n    draw.text((24,45),\'LABEL A-104\',font=font,fill=20)\n    draw.text((24,90),\'BATCH 2026\',font=font,fill=20)\n    x=torch.from_numpy(np.asarray(im).copy()).float()/127.5-1\n    return x[None,None].repeat(batch,3,1,1).to(device)\n\n\ndef save_preview(x,reconstructed,destination):\n    from PIL import Image,ImageDraw\n    import numpy as np\n    images=[]\n    for tensor in [x,reconstructed]:\n        array=((tensor[0].detach().float().cpu().permute(1,2,0).numpy()+1)/2*255).clip(0,255).astype(np.uint8)\n        images.append(Image.fromarray(array))\n    sheet=Image.new(\'RGB\',(520,302),\'white\');draw=ImageDraw.Draw(sheet)\n    draw.text((12,10),\'INTEGRATION CHECK ONLY - TWO UPDATES\',fill=\'darkred\')\n    draw.text((12,32),\'Synthetic input\',fill=\'black\');draw.text((268,32),\'8-step reconstruction\',fill=\'black\')\n    for i,im in enumerate(images):sheet.paste(im.resize((224,224)),(12+i*256,54))\n    draw.text((12,283),\'No anomaly-removal or accuracy claim.\',fill=\'darkred\');sheet.save(destination)\n\n\ndef run(output,cache=None):\n    import importlib.metadata\n    import numpy as np\n    import torch\n    import numba\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if not torch.cuda.is_available():raise RuntimeError(\'Select a GPU accelerator before running this notebook\')\n    cache=Path(cache) if cache else output/\'upstream\'/COMMIT\n    hashes=fetch_sources(cache)\n    random.seed(230224);np.random.seed(230224);torch.manual_seed(230224)\n    numba.set_num_threads(min(2,numba.get_num_threads()))\n    module,ns,adapter_type,patches=load_author_components(cache,output)\n    config={\'img_size\':224,\'patch_size\':16,\'in_chans\':3,\'embed_dim\':384,\'depth\':12,\n            \'num_heads\':6,\'mlp_ratio\':4.,\'num_classes\':None,\'mlp_time_embed\':True,\n            \'use_dec\':[\'DAFF\',\'DAFF\',\'DAFF\'],\'PE_type\':\'SPE\',\'refinement\':True,\'qkv_bias\':False}\n    report={\'status\':\'running\',\'scope\':\'integration_check_only\',\'upstream_commit\':COMMIT,\n            \'source_sha256\':hashes,\'compatibility_changes\':patches,\'model_configuration\':config,\n            \'configuration_note\':\'Illustrated SPE/DMHA/HFF/refinement variant. Code depth=12 gives six encoder blocks, one middle, six decoder blocks. This is not asserted to match every paper table.\',\n            \'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0),\n            \'gpu_vram_gib\':torch.cuda.get_device_properties(0).total_memory/2**30,\n            \'packages\':{n:importlib.metadata.version(n) for n in [\'timm\',\'einops\',\'numba\',\'numpy\']},\n            \'noise_parameters\':{\'octave\':6,\'frequency\':64,\'persistence\':.9},\n            \'training_steps\':2,\'batch_size\':2,\'total_diffusion_steps\':1000,\'reconstruction_steps\':8}\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    device=torch.device(\'cuda:0\');torch.cuda.reset_peak_memory_stats()\n    print(\'Building author DTU-Net, width 384, six attention heads...\',flush=True)\n    backbone=module.UDHVT(**config).to(device);model=adapter_type(backbone)\n    report[\'parameter_count\']=sum(p.numel() for p in model.parameters())\n    x=synthetic_batch(224,2,device)\n    t=torch.tensor([50,150],device=device,dtype=torch.long)\n    diffusion=ns[\'GaussianDiffusionModel\']([224,224],ns[\'get_beta_schedule\'](1000,\'cosine\'),img_channels=3,\n                 loss_type=\'l2\',noise=\'4dsimplex\',octave=6,frequency=64,persistence=.9,train=False)\n    print(\'Compiling the author 4D noise function on CPU; first use may take a few minutes...\',flush=True)\n    start=time.perf_counter()\n    probe=torch.zeros(1,1,4,4,device=device)\n    diffusion.noise_fn(probe,torch.tensor([5],device=device))\n    report[\'noise_first_compile_seconds\']=time.perf_counter()-start\n    noise=diffusion.noise_fn(x,t).float()\n    assert noise.shape==x.shape and torch.isfinite(noise).all()\n    assert not torch.allclose(noise[0],noise[1]),\'Different time coordinates unexpectedly generated identical samples\'\n    report[\'noise_shape\']=list(noise.shape)\n    report[\'noise_mean\']=float(noise.mean());report[\'noise_std\']=float(noise.std())\n    report[\'noise_normalization\']=\'Author raw amplitude retained; no per-sample standardization.\'\n    report[\'batch_noise_note\']=\'The author generator uses t as the fourth coordinate. Duplicate time coordinates with one seed can produce identical noise across batch entries; this remains to be assessed during training.\'\n    model.train();optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.)\n    report[\'losses\']=[];report[\'gradient_norms\']=[]\n    tracked=backbone.pos_embed.detach().clone()\n    for step in range(2):\n        optimizer.zero_grad(set_to_none=True)\n        losses,noisy,predicted=diffusion.calc_loss(model,x,None,t)\n        loss=losses[\'loss\'].mean()\n        assert predicted.shape==x.shape and torch.isfinite(loss)\n        loss.backward()\n        grads=[p.grad for p in model.parameters() if p.grad is not None]\n        assert grads and all(torch.isfinite(g).all() for g in grads)\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        optimizer.step()\n        report[\'losses\'].append(float(loss.detach()))\n        report[\'gradient_norms\'].append(float(norm))\n        print(f\'Update {step+1}/2 passed; L2 noise loss {float(loss.detach()):.6f}\',flush=True)\n    assert not torch.equal(tracked,backbone.pos_embed.detach()),\'Optimizer did not change the tracked parameter\'\n    report[\'tracked_parameter_changed\']=True\n    report[\'parameters_without_grad\']=[name for name,p in model.named_parameters() if p.grad is None]\n    model.eval()\n    print(\'Checking eight author reverse-diffusion steps. This model is not trained for detection.\',flush=True)\n    with torch.no_grad():\n        result=diffusion.forward_backward(model,x[:1],None,see_whole_sequence=None,t_distance=8,denoise_fn=\'noise_fn\')\n    assert result.shape==x[:1].shape and torch.isfinite(result).all()\n    residual=(x[:1]-result).square().mean(dim=1)\n    assert residual.shape==(1,224,224) and torch.isfinite(residual).all()\n    save_preview(x,result,output/\'integration_preview.png\')\n    report[\'reconstruction_shape\']=list(result.shape);report[\'residual_shape\']=list(residual.shape)\n    report[\'peak_gpu_allocated_gib\']=torch.cuda.max_memory_allocated()/2**30\n    report[\'peak_gpu_reserved_gib\']=torch.cuda.max_memory_reserved()/2**30\n    report[\'status\']=\'passed\'\n    report[\'not_completed\']=[\'Training a useful anomaly model\',\'Real label dataset\',\'Paper metrics reproduction\',\'Quality comparison with CPU baseline\']\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'\\nDTU-NET + TSIMPLEX INTEGRATION CHECK PASSED\',flush=True)\n    print(json.dumps({k:report[k] for k in [\'parameter_count\',\'noise_shape\',\'losses\',\'reconstruction_shape\',\'peak_gpu_allocated_gib\',\'peak_gpu_reserved_gib\']},indent=2))\n    print(\'Saved:\',output/\'integration_report.json\',flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',default=\'artifacts/author_integration\')\n    parser.add_argument(\'--cache\',default=None)\n    args=parser.parse_args();run(args.output,args.cache)\n', encoding='utf8')
print('Wrote:', script)


In [ ]:
output = work / 'artifacts' / 'author_integration'
subprocess.run([sys.executable, '-u', str(script), '--output', str(output)], check=True)


In [ ]:
report = json.loads((output / 'integration_report.json').read_text())
print(json.dumps(report, indent=2))
from IPython.display import display, Image
display(Image(filename=str(output / 'integration_preview.png')))
import shutil
archive = shutil.make_archive(str(work / 'author_integration_results'), 'zip', output)
print('Download this result bundle:', archive)


## What passing means

The selected architecture, noise, gradient update, and short reconstruction execute together on this runtime. It does not establish segmentation accuracy, speed at the full paper batch size, or reproducibility of every author variant. The preview is explicitly labelled as an integration check.

Save `integration_report.json`. Next: a resumable training pipeline, dataset split, validation protocol, and a limited benchmark experiment before the printed-label extension. No threshold from this check should be used on real photographs.